# 🩺 Módulo de Detecção Inteligente de Sangramentos (Treino)
Este notebook realiza o **treino do modelo para análise de vídeos em tempo real** para identificar episódios de sangramento em um contexto de cirurgias ginecológicas laparoscópicas.

*   **Arquitetura:** Implementado com o modelo YOLOv8 para processamento de visão computacional de alta performance.
*   **Dataset:** Treinado utilizando o conjunto de dados especializado GynSurg.
*   **Objetivo:** Atender ao requisito de detecção automatizada de anomalias, otimizando a agilidade e a precisão técnica no suporte à decisão de equipes cirúrgicas.

⚠️ **Aviso de Responsabilidade (Disclaimer):**
Este projeto foi desenvolvido exclusivamente para fins educacionais. Ressaltamos que a ferramenta não substitui validações médicas e não está apta para embasar decisões clínicas, realizar diagnósticos, apoiar triagens ou auxiliar vítimas em situações de emergência. Toda e qualquer interpretação dos resultados deve ficar a cargo de profissionais competentes.

# 🛠️ Configuração de Ambiente e Sincronização com AWS S3

Este bloco de código prepara o ambiente do Google Colab para interagir com a infraestrutura da AWS. O processo é dividido em três etapas críticas:

**1. Preparação do Ambiente (Instalação):**

Executamos **!pip install awscli -q**. Este comando instala a interface de linha de comando oficial da AWS no ambiente virtual do Colab. O sinalizador **-q (quiet)** é utilizado para manter o log de saída limpo, omitindo mensagens desnecessárias durante a instalação.

**2. Segurança e Autenticação:**

*   Utilizamos a biblioteca google.colab.userdata para recuperar chaves de acesso sensíveis armazenadas de forma segura no cofre de segredos do seu projeto.
*   As variáveis **AWS_ACCESS_KEY_ID** e **AWS_SECRET_ACCESS_KEY** são injetadas nas variáveis de ambiente do sistema (os.environ). Isso permite que o aws-cli autentique suas requisições automaticamente sem que você precise expor suas credenciais diretamente no código.
*   Definimos **AWS_DEFAULT_REGION** como **sa-east-1 (São Paulo)** para otimizar a latência e garantir que a comunicação seja direcionada à região correta do *bucket*.

**3. Transferência de Dados:**

*   Utilizamos o comando **!aws s3 sync**. Diferente de um download simples (cp), o comando sync é ideal para datasets, pois ele apenas copia os arquivos que são novos ou que foram alterados no bucket de origem (**s3://dataset-gynsurg-bleeding/dataset_yolo/**) para o diretório local (**/content/dataset_yolo/**).
*   Ao final, o sistema imprime uma confirmação de sucesso, garantindo que o dataset está pronto para o processamento do modelo YOLO.
*   Esta estrutura facilita a manutenção do pipeline e garante que as melhores práticas de segurança (não *hardcoding* de chaves) sejam seguidas.

In [ ]:
# 1. Instala o AWS CLI no Colab
!pip install awscli -q

import os
from google.colab import userdata

# 2. Carrega as chaves do Secrets do Colab
os.environ["AWS_ACCESS_KEY_ID"] = userdata.get('AWS_ACCESS_KEY_ID')
os.environ["AWS_SECRET_ACCESS_KEY"] = userdata.get('AWS_SECRET_ACCESS_KEY')
os.environ["AWS_DEFAULT_REGION"] = "sa-east-1"

print("Baixando dataset do S3...")
# 3. Faz o download do dataset do seu bucket para a pasta /content/dataset_yolo/
!aws s3 sync s3://dataset-gynsurg-bleeding/dataset_yolo/ /content/dataset_yolo/
print("Download concluído com sucesso!")

A saída de streaming foi truncada nas últimas 5000 linhas.
download: s3://dataset-gynsurg-bleeding/dataset_yolo/train/bleeding/case_409_Bleeding_0_00_15.166667-0_00_56.03333330fps_clip_307_frame_0001.jpg to dataset_yolo/train/bleeding/case_409_Bleeding_0_00_15.166667-0_00_56.03333330fps_clip_307_frame_0001.jpg
download: s3://dataset-gynsurg-bleeding/dataset_yolo/train/bleeding/case_409_Bleeding_0_00_15.166667-0_00_56.03333330fps_clip_307_frame_0003.jpg to dataset_yolo/train/bleeding/case_409_Bleeding_0_00_15.166667-0_00_56.03333330fps_clip_307_frame_0003.jpg
download: s3://dataset-gynsurg-bleeding/dataset_yolo/train/bleeding/case_409_Bleeding_0_00_15.166667-0_00_56.03333330fps_clip_308_frame_0000.jpg to dataset_yolo/train/bleeding/case_409_Bleeding_0_00_15.166667-0_00_56.03333330fps_clip_308_frame_0000.jpg
download: s3://dataset-gynsurg-bleeding/dataset_yolo/train/bleeding/case_409_Bleeding_0_00_15.166667-0_00_56.03333330fps_clip_308_frame_0002.jpg to dataset_yolo/train/bleeding/case_4

# 🚀 **Treinamento do Modelo de Classificação YOLOv8**

Este bloco é responsável por preparar e executar o treinamento do modelo de rede neural, adaptando-o para a tarefa específica de classificação de imagens no seu dataset.

**Instalação de Dependências:**

*   O comando **!pip install ultralytics -q** instala o ecossistema completo da Ultralytics. O parâmetro -q suprime a saída detalhada, mantendo o notebook limpo.

**Inicialização do Modelo:**

*   Importamos a classe YOLO e instanciamos o modelo **yolov8n-cls.pt**. A escolha da variante Nano **(-n)** é estratégica: ela prioriza a velocidade e a leveza, sendo ideal para inferências em tempo real, enquanto o sufixo **-cls** indica que a arquitetura foi otimizada especificamente para classificação de imagens.

**Configuração e Execução do Treinamento:**

*   O método **model.train()** é configurado com os seguintes hiperparâmetros essenciais:

    *   **data:** Aponta para o diretório raiz das suas imagens (onde as subpastas devem estar organizadas por classes).

    *   **epochs=25:** Define que o modelo passará pelo dataset completo 25 vezes para aprender os padrões.

    *   **imgsz=224:** Redimensiona as imagens de entrada para 224x224 pixels, padrão comum para modelos de classificação eficientes.

    *   **batch=32:** Processa 32 imagens simultaneamente, equilibrando o uso de memória da GPU e a estabilidade do gradiente.

*   **Organização de Resultados:** O uso dos argumentos **project** e **name** garante que todos os logs, pesos finais **(best.pt)** e métricas de desempenho (matriz de confusão, gráficos de perda/loss) sejam salvos organizadamente no diretório **gynsurg_project/bleeding_model/**.

Esta configuração garante uma estrutura de trabalho reprodutível e facilita a comparação de diferentes experimentos conforme ajustes nos hiperparâmetros do modelo.

In [ ]:
# Instala a biblioteca do YOLOv8
!pip install ultralytics -q

from ultralytics import YOLO

# Carrega o modelo pré-treinado focado em classificação (sufixo -cls)
# 'yolov8n-cls.pt' é o modelo Nano (mais rápido e leve)
model = YOLO('yolov8n-cls.pt')

print("Iniciando o treinamento...")
# Inicia o treinamento
results = model.train(
    data='/content/dataset_yolo', # Caminho da pasta principal do dataset
    epochs=25,                    # Número de épocas (ajuste conforme necessário)
    imgsz=224,                    # Tamanho da imagem para classificação
    batch=32,                     # Tamanho do lote
    project='gynsurg_project',    # Nome da pasta onde salvará os resultados
    name='bleeding_model'         # Nome da execução atual
)

Iniciando o treinamento...
Ultralytics 8.4.60 🚀 Python-3.12.13 torch-2.11.0+cpu CPU (Intel Xeon CPU @ 2.20GHz)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/dataset_yolo, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=25, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=224, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n-cls.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=bleeding_model-2, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto

# ☁️ **Persistência de Dados: Exportação do Modelo para o S3**

Após a conclusão do ciclo de treinamento, é fundamental garantir que os pesos (*model weights*) e os logs de desempenho gerados pelo YOLOv8 sejam armazenados de forma persistente, evitando a perda de dados caso a sessão do Google Colab seja encerrada.

*   **Sincronização de Artefatos:**

    *   O comando **!aws s3 sync** é utilizado novamente para realizar uma cópia sincronizada da pasta local /content/runs/classify/gynsurg_project/bleeding_model-2 para o bucket S3 de destino (s3://dataset-gynsurg-bleeding/resultados_treinamento/).

    *   Este diretório local contém os arquivos essenciais do treinamento, incluindo o arquivo **best.pt** (os pesos com a melhor acurácia obtida) e os logs de métricas (gráficos de perda, matriz de confusão etc.).

*   **Vantagem do sync:**

    *   Ao utilizar o comando sync em vez de uma cópia simples, garantimos que apenas os arquivos novos ou modificados sejam transferidos, otimizando o tempo de execução e o consumo de banda de rede.

*   **Conclusão do Workflow:**

    *   Após o comando, a mensagem de sucesso confirma que o modelo está agora centralizado e disponível na nuvem, permitindo que ele seja baixado posteriormente para ambientes de produção ou servidores de inferência.

In [ ]:
print("Salvando o modelo treinado de volta no S3...")
# Sincroniza a pasta padrão gerada pelo YOLOv8
!aws s3 sync /content/runs/classify/gynsurg_project/bleeding_model-2 s3://dataset-gynsurg-bleeding/resultados_treinamento/
print("Upload dos pesos concluído!")

Salvando o modelo treinado de volta no S3...
upload: runs/classify/gynsurg_project/bleeding_model-2/confusion_matrix.png to s3://dataset-gynsurg-bleeding/resultados_treinamento/confusion_matrix.png
upload: runs/classify/gynsurg_project/bleeding_model-2/args.yaml to s3://dataset-gynsurg-bleeding/resultados_treinamento/args.yaml
upload: runs/classify/gynsurg_project/bleeding_model-2/results.csv to s3://dataset-gynsurg-bleeding/resultados_treinamento/results.csv
upload: runs/classify/gynsurg_project/bleeding_model-2/val_batch0_labels.jpg to s3://dataset-gynsurg-bleeding/resultados_treinamento/val_batch0_labels.jpg
upload: runs/classify/gynsurg_project/bleeding_model-2/train_batch2280.jpg to s3://dataset-gynsurg-bleeding/resultados_treinamento/train_batch2280.jpg
upload: runs/classify/gynsurg_project/bleeding_model-2/train_batch1.jpg to s3://dataset-gynsurg-bleeding/resultados_treinamento/train_batch1.jpg
upload: runs/classify/gynsurg_project/bleeding_model-2/results.png to s3://dataset-gy